# CVP mid-term: train vanilla 3DGS (gsplat) on RealX3D and score its geometry

Works on **Google Colab** (Runtime → Change runtime type → **T4 GPU**) and **Kaggle** (Settings → Accelerator **GPU T4**, and **Internet ON**).

Run the cells top to bottom. Expected times on a free T4:
- install + compile gsplat: ~5–10 min (first time only per session)
- data download (one scene, 2 conditions + laser scan): ~2–4 min
- training 30k steps: roughly 45–90 min per condition (7k steps: ~10–20 min)
- geometry evaluation: ~3–5 min

At the end, download `cvp_results.zip` (all metrics, previews, ply, figures).

In [1]:
# 1) GPU check
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv

name, compute_cap, memory.total [MiB]
Tesla T4, 7.5, 15360 MiB


In [6]:
# 2) Get the code. EITHER clone from GitHub (after you pushed it) ...
import os
REPO_URL = "https://github.com/Saksham-Bali/cvp_project"   # <- your repo
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.chdir(WORK)
if not os.path.exists("cvp_project"):
    !git clone -q $REPO_URL
# If the repo is PRIVATE, use https://<your-github-token>@github.com/Saksham-Bali/cvp_project
# ... OR upload cvp_code.zip with the file browser and run:  !unzip -q cvp_code.zip -d cvp_project
os.chdir(f"{WORK}/cvp_project")
!ls

cvp   notebooks  README.md	       requirements-train.txt  scripts
docs  outputs	 requirements-cpu.txt  RUN_MIDTERM.md	       tests


In [ ]:
# 3) Install (gsplat compiles its CUDA code on first use; this cell triggers that compile now)
!pip -q install gsplat==1.5.3 ninja jaxtyping rich plyfile huggingface_hub requests lpips
!python scripts/gsplat_smoke.py
# If the compile fails on a new PyTorch: !pip install git+https://github.com/nerfstudio-project/gsplat.git  (then re-run this cell)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 4.8 MB/s eta 0:00:00
( ●    ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(  ●   ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(   ●  ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(    ● ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(     ●) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(    ● ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(   ●  ) gsplat: Setting up CUDA with MAX_JOBS=10 (This may take a few minutes 
(  ●   ) gsplat: Setting up CUDA with MAX_JOBS

In [ ]:
# 4) Download ONE scene: quarter-res images for the 2 strong conditions + laser ground truth
#    (+ RealX3D's published 3DGS numbers for the same scene, for the PSNR comparison)
SCENE = "Ujikintoki"
!python scripts/download_realx3d.py --what quarter gt --scenes $SCENE --conditions motion_strong defocus_strong
!python scripts/download_realx3d.py --what baselines --scenes $SCENE --conditions motion_strong

In [ ]:
# 5) Train vanilla 3DGS on STRONG MOTION blur.  STEPS=30000 for the real run, 7000 for a quick one.
STEPS = 30000
!python scripts/train_gsplat.py --data data --scene $SCENE --condition motion_strong --steps $STEPS

In [ ]:
# 6) Score its geometry against the laser scan, overall and per structure type
!python scripts/eval_run.py --run runs/$SCENE/motion_strong/vanilla/s0 --data data

In [ ]:
# 7) (if time) the same for STRONG DEFOCUS
!python scripts/train_gsplat.py --data data --scene $SCENE --condition defocus_strong --steps $STEPS
!python scripts/eval_run.py --run runs/$SCENE/defocus_strong/vanilla/s0 --data data

In [ ]:
# 8) (optional, if time) SHARP CONTROL: same views but the sharp reference images
!python scripts/train_gsplat.py --data data --scene $SCENE --condition motion_strong --images sharp --steps $STEPS
!python scripts/eval_run.py --run runs/$SCENE/motion_strong/sharp_control/s0 --data data

In [ ]:
# 9) Look at a preview: blurred input | our render | sharp reference | rendered depth
import glob
from IPython.display import Image, display
for p in sorted(glob.glob(f"runs/{SCENE}/*/*/s0/previews/*.jpg"))[:4]:
    print(p); display(Image(p, width=1100))

In [ ]:
# 10) Zip everything small enough to download (metrics, previews, geometry, ply)
!zip -q -r cvp_results.zip runs outputs -x "*/depth/*"
!ls -lh cvp_results.zip
# Colab: download from the file browser (left), or:
try:
    from google.colab import files; files.download("cvp_results.zip")
except Exception:
    print("On Kaggle: find cvp_results.zip under Output (/kaggle/working)")